# LoRA 角色扮演微调 — Colab 版

**模型:** Qwen2.5-1.5B-Instruct + QLoRA 4-bit  
**数据集:** hieunguyenminh/roleplay (~5.7K 条)  
**显存:** T4 15GB 绰绰有余

特性:
- 模型和日志自动保存到 Google Drive
- 断点续传：中断后重新运行训练 cell 即可恢复
- 训练结束自动生成 loss 曲线

## 1. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive 中的保存路径（可按需修改）
import os
DRIVE_DIR = '/content/drive/MyDrive/roleplay-lora'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive 保存路径: {DRIVE_DIR}')

## 2. 安装依赖

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes tensorboard matplotlib

## 3. 验证 GPU

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  显存: {props.total_memory / (1024**3):.1f} GB')
else:
    raise RuntimeError('请通过 运行时 → 更改运行时类型 → 选择 T4 GPU')

## 4. 配置参数

In [ ]:
import os
from datetime import datetime

# ========== 模型配置 ==========
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# ========== 数据集配置 ==========
DATASET_NAME = "hieunguyenminh/roleplay"
MAX_SEQ_LENGTH = 768  # T4 15GB 可以用更大长度

# ========== QLoRA 配置 ==========
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ========== 训练配置 ==========
BATCH_SIZE = 2               # T4 显存更大，可以开大 batch
GRADIENT_ACCUMULATION = 4    # 等效 batch = 2 × 4 = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_STEPS = 50
LOGGING_STEPS = 5
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2

# ========== 路径配置 ==========
OUTPUT_DIR = os.path.join(DRIVE_DIR, "output")
LOG_DIR = os.path.join(DRIVE_DIR, "logs", datetime.now().strftime("%Y%m%d-%H%M%S"))

print(f'Drive 目录: {DRIVE_DIR}')
print(f'输出目录: {OUTPUT_DIR}')
print(f'日志目录: {LOG_DIR}')

## 5. 加载并格式化数据集

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Tokenizer vocab: {len(tokenizer)}')

# 加载数据集
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f'数据集: {len(raw_dataset)} 条  |  字段: {raw_dataset.column_names}')
print(f'\n--- 样本预览 ---')
print(raw_dataset[0]['text'][:300])

In [ ]:
import re

def format_conversation(example):
    """
    数据集格式: <|system|>角色描述</s><|user|>消息</s><|assistant|>回复</s>
    转为 Qwen2.5 ChatML (<|im_start|>...<|im_end|>)
    """
    text = example["text"]
    pattern = r"<\|(system|user|assistant)\|>(.*?)</s>"
    matches = re.findall(pattern, text, re.DOTALL)

    messages = []
    for role, content in matches:
        content = content.strip()
        if content:
            messages.append({"role": role, "content": content})

    if not messages:
        return {"text": ""}

    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": formatted}


print("正在格式化...")
formatted_dataset = raw_dataset.map(format_conversation, remove_columns=raw_dataset.column_names)
formatted_dataset = formatted_dataset.filter(lambda x: len(x["text"]) > 0)

sample = formatted_dataset[0]["text"]
print(f'格式化样本数: {len(formatted_dataset)}')
print(f'\n--- Qwen2.5 ChatML 格式（前 400 字符）---')
print(sample[:400])

In [ ]:
# 划分 train/eval
split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f'训练集: {len(train_dataset)} 条  |  验证集: {len(eval_dataset)} 条')
print(f'等效 batch size: {BATCH_SIZE} × {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION}')
print(f'预计步数/epoch: ~{len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION)}')

## 6. 加载模型 (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f'正在加载: {MODEL_NAME} ...')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache = False

print(f'显存: {torch.cuda.memory_allocated() / (1024**3):.2f} GB')
print(f'参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')

## 7. 配置 LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'可训练参数: {trainable / 1e6:.2f}M / {total / 1e6:.0f}M ({100 * trainable / total:.2f}%)')
print(f'LoRA: r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES}')

## 8. 训练

支持断点续传：中断后重新运行本 cell 即可从 Drive 中最新 checkpoint 恢复。

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

# ---- 检查 Drive 中是否有 checkpoint 可恢复 ----
resume_from_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [
        d for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
    ]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[1]))
        resume_from_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])
        print(f'🔁 从 Drive 恢复: {resume_from_checkpoint}')
    else:
        print('未发现 checkpoint，开始全新训练')
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print('未发现 checkpoint，开始全新训练')

# ---- 训练参数 ----
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    logging_dir=LOG_DIR,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_8bit",
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
)

print(f'TensorBoard: tensorboard --logdir {LOG_DIR}')

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    formatting_func=lambda x: x["text"],
)

if resume_from_checkpoint:
    print(f'从 {resume_from_checkpoint} 恢复训练...')
else:
    print('开始全新训练...')

trainer.train(resume_from_checkpoint=resume_from_checkpoint)

print('\n✅ 训练完成！')

## 9. 保存模型到 Drive

In [ ]:
final_model_path = os.path.join(DRIVE_DIR, "final-lora-adapter")
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

adapter_size = sum(f.stat().st_size for f in os.scandir(final_model_path) if f.is_file()) / 1e6
print(f'LoRA adapter 已保存到 Drive: {final_model_path}')
print(f'Adapter 大小: {adapter_size:.1f} MB')

del trainer
torch.cuda.empty_cache()
print('训练状态已清理')

## 10. 可视化训练指标

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

# 找到 TensorBoard event 文件
event_files = sorted(glob.glob(os.path.join(LOG_DIR, "**/events.out.tfevents.*"), recursive=True))
if not event_files:
    event_files = sorted(glob.glob(os.path.join(LOG_DIR, "events.out.tfevents.*")))

if not event_files:
    print("未找到 TensorBoard 日志")
else:
    event_file = event_files[0]
    print(f'日志: {event_file}')
    
    ea = EventAccumulator(event_file)
    ea.Reload()
    
    if "train/loss" not in ea.Tags()["scalars"]:
        print(f'可用 scalars: {ea.Tags()["scalars"]}')
    else:
        train_loss = ea.Scalars("train/loss")
        steps = [s.step for s in train_loss]
        values = [s.value for s in train_loss]
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 左: 原始 Loss
        axes[0].plot(steps, values, "b-", alpha=0.7, linewidth=1)
        axes[0].scatter(steps, values, s=15, alpha=0.5, color="blue")
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training Loss")
        axes[0].grid(True, alpha=0.3)
        
        # 右: 平滑 Loss
        axes[1].plot(steps, values, "b-", alpha=0.25, linewidth=0.8, label="Raw")
        if len(values) >= 5:
            w = min(5, len(values))
            smoothed = np.convolve(values, np.ones(w)/w, mode="valid")
            axes[1].plot(steps[w-1:], smoothed, "r-", linewidth=1.5, label=f"Smoothed (w={w})")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Smoothed Loss")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        loss_png = os.path.join(DRIVE_DIR, "loss_curve.png")
        plt.savefig(loss_png, dpi=150, bbox_inches="tight")
        plt.show()
        print(f'Loss 曲线已保存: {loss_png}')
        print(f'Loss: {values[0]:.4f} → {values[-1]:.4f} (↓ {values[0]-values[-1]:.4f})')

In [ ]:
# per-epoch 可视化
if event_files and 'train/loss' in ea.Tags()['scalars']:
    steps_per_epoch = max(len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION), 1)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    epochs_arr = [s / steps_per_epoch for s in steps]
    ax.plot(epochs_arr, values, 'b-', alpha=0.6, linewidth=1)
    ax.scatter(epochs_arr, values, s=10, alpha=0.4, color='blue')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training Loss by Epoch')
    ax.grid(True, alpha=0.3)
    
    for ep in range(1, NUM_EPOCHS + 1):
        ax.axvline(x=ep, color='red', linestyle='--', alpha=0.3, linewidth=1)
    
    plt.tight_layout()
    epoch_png = os.path.join(DRIVE_DIR, 'loss_by_epoch.png')
    plt.savefig(epoch_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Epoch 曲线已保存: {epoch_png}')

## 11. 推理测试

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

if "model" not in dir() or model is None:
    print("重新加载模型...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(model, final_model_path)

model.eval()

test_character = """你叫艾尔文，是一位隐居森林的精灵药剂师，已经活了300年。
你说话温和又略带古风，偶尔会引用植物的名字来比喻人生道理。
你对人类感到好奇，因为他们寿命虽短却充满热情。"""

messages = [
    {"role": "system", "content": test_character},
    {"role": "user", "content": "请问你有什么药可以治疗失眠？"},
]

print("正在生成...")
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print(f"\n{'='*50}")
print(f"角色: 精灵药剂师 艾尔文")
print(f"用户: 请问你有什么药可以治疗失眠？")
print(f"\n模型回复:\n{response}")
print(f"{'='*50}")

---

## 附录

### Drive 目录结构

```
MyDrive/roleplay-lora/
├── output/                  # 训练 checkpoint
│   ├── checkpoint-100/
│   ├── checkpoint-200/
│   └── ...
├── logs/                    # TensorBoard 日志
├── final-lora-adapter/      # 最终 LoRA 权重
├── loss_curve.png
└── loss_by_epoch.png
```

### 断点续传

Colab 断开后，重新运行所有 cell 即可。训练 cell 会自动检测 Drive 中的 `checkpoint-*` 并从最新进度恢复。

### 从 Drive 加载模型推理

```python
from peft import PeftModel
model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/roleplay-lora/final-lora-adapter')
```